In [ ]:
import pandas as pd
import numpy as np
from gensim.models import FastText
from nltk.tokenize import word_tokenize
import nltk
import umap
import plotly.express as px


c:\Users\ADRIA\OneDrive\Escritorio\Descargar_datos\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [3]:
df = pd.read_parquet("../parquet/resenas_limpias.parquet")

In [4]:
df.sample(5)

,resena_id,juego_id,titulo,resena_texto,recomendado,votos_utiles,votos_graciosos,puntuacion_ponderada,minutos_al_resenar,minutos_totales,fecha_creacion_unix,autor_num_resenas,recibido_gratis,escrito_acceso_anticipado,resena_limpia
24386,103439763,1369,New World: Aeternum,"El juego esta bueno, mejor si se juega con ami...",1,0,0,0.476190,8467,25392.0,1637780109,11,0,0,"el juego esta bueno, mejor si se juega con ami..."
240726,74388032,1529,Quake Champions,excelente recomendado desde quake 3 arena aunq...,1,1,0,0.512137,450,642.0,1597559428,66,0,1,excelente recomendado desde quake 3 arena aunq...
252908,21089963,517,Metal Slug,Infancia jugando estos en emuladores gracias a...,1,0,0,0.500000,345,595.0,1455064456,13,0,0,infancia jugando estos en emuladores gracias a...
213155,138207806,911,Sunset Overdrive,"Es un juego infravalorado, super divertido y c...",1,0,0,0.500000,105,194.0,1683932062,101,0,0,"es un juego infravalorado, super divertido y c..."
265159,101438057,4328,Zombie Driver HD,"Sinceramente no esperaba mucho de este juego, ...",1,0,0,0.500000,879,879.0,1634909453,218,0,0,"sinceramente no esperaba mucho de este juego, ..."


In [5]:
corpus = df['resena_limpia'].tolist()


In [6]:
corpus

['juego lleno de chiteros, con una comunidad de jugadores muy expresion_negativa_fuerte. jueguen otros juegos.',
 'muy buen juego free to play pero si queres jugar competitivo no tener de otra de comprarte el premier',
 'muchos peruanos y argentinos q lloran jugando a una partida random sin rango, te amenazan con matar a tu familia si no matas a los otros 5 y casualmente siempre son los primeros en morir, no recomiendo igual de bosta q el togeder',
 'es complicado jugar cuando te toca un equipo de bots y vos sos tan bueno q te cargas el equipo al hombro, bah no se eso me pasa siempre.',
 'es buen juego, pero esta lleno de chiteros! el sistema de vac es nulo',
 'el juego es re inclusivo, todos los de mi equipo tienen down',
 'que juego de gran belliisima, muy buen ueo 0 bugs, el mejor anticheat y el mejor delay de diparos a la latencia, calificacion_10_10',
 'xiteros de expresion_negativa_fuerte se ragebaitean al toque y arruinan las partidas',
 'muy buen juego, buenos graficos, buenos 

In [7]:
def tokenizar(texto):
    return word_tokenize(texto, language='spanish')

In [8]:
corpus_tokenizado = [tokenizar(t) for t in corpus]


In [9]:
corpus_tokenizado

[['juego',
  'lleno',
  'de',
  'chiteros',
  ',',
  'con',
  'una',
  'comunidad',
  'de',
  'jugadores',
  'muy',
  'expresion_negativa_fuerte',
  '.',
  'jueguen',
  'otros',
  'juegos',
  '.'],
 ['muy',
  'buen',
  'juego',
  'free',
  'to',
  'play',
  'pero',
  'si',
  'queres',
  'jugar',
  'competitivo',
  'no',
  'tener',
  'de',
  'otra',
  'de',
  'comprarte',
  'el',
  'premier'],
 ['muchos',
  'peruanos',
  'y',
  'argentinos',
  'q',
  'lloran',
  'jugando',
  'a',
  'una',
  'partida',
  'random',
  'sin',
  'rango',
  ',',
  'te',
  'amenazan',
  'con',
  'matar',
  'a',
  'tu',
  'familia',
  'si',
  'no',
  'matas',
  'a',
  'los',
  'otros',
  '5',
  'y',
  'casualmente',
  'siempre',
  'son',
  'los',
  'primeros',
  'en',
  'morir',
  ',',
  'no',
  'recomiendo',
  'igual',
  'de',
  'bosta',
  'q',
  'el',
  'togeder'],
 ['es',
  'complicado',
  'jugar',
  'cuando',
  'te',
  'toca',
  'un',
  'equipo',
  'de',
  'bots',
  'y',
  'vos',
  'sos',
  'tan',
  'bueno'

In [10]:
modelo_ft = FastText(
    sentences=corpus_tokenizado,
    vector_size=200,
    window=5,
    min_count=3,
    workers=10,
    epochs=10,
    sg=1,seed=31,
)

In [11]:
modelo_ft.save("../models/resenas_fastest.model")

In [12]:
def vector_resena(tokens):
    vecs = [modelo_ft.wv[t] for t in tokens if t in modelo_ft.wv]
    if not vecs:
        return np.zeros(modelo_ft.vector_size)
    return np.mean(vecs, axis=0)

In [13]:
embeddings = np.array([
    vector_resena(tokenizar(t)) for t in corpus
])

In [14]:
np.save("../datos/resenas_vectorizadas.npy", embeddings)

In [15]:
print(len(corpus))
print(len(modelo_ft.wv))
print(embeddings.shape)

295974
79548
(295974, 200)


In [16]:
reducer = umap.UMAP(
    n_components=5,
    n_neighbors=30,
    min_dist=0.0,
    metric='cosine',
    random_state=31,
)

In [17]:
embeddings_umap = reducer.fit_transform(embeddings)
 
 

c:\Users\ADRIA\OneDrive\Escritorio\Descargar_datos\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Shape original:  (296402, 100)

In [18]:
print(embeddings.shape)
print(embeddings_umap.shape)

(295974, 200)
(295974, 5)


In [19]:
import pandas as pd
import numpy as np
import hdbscan


clusterer = hdbscan.HDBSCAN(
    min_cluster_size=2000,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom',
)

In [20]:
labels = clusterer.fit_predict(embeddings_umap)

In [21]:
df['cluster'] = labels



In [22]:
df['cluster'].value_counts()

cluster
-1    232313
 5     17251
 2     11977
 4      7023
 3      6415
 6      5269
 0      4886
 7      4078
 8      3650
 1      3112
Name: count, dtype: int64

In [23]:
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_ruido    = (labels == -1).sum()

In [24]:

print(f"Clusters encontrados: {n_clusters}")
print(f"Reseñas sin cluster (ruido): {n_ruido:,} ({n_ruido/len(labels)*100:.1f}%)")
print(f"\nTamaño por cluster:")
print(pd.Series(labels).value_counts().sort_index().rename(index={-1: 'ruido'}))


Clusters encontrados: 9
Reseñas sin cluster (ruido): 232,313 (78.5%)

Tamaño por cluster:
ruido    232313
0          4886
1          3112
2         11977
3          6415
4          7023
5         17251
6          5269
7          4078
8          3650
Name: count, dtype: int64


In [25]:
df.to_parquet("../parquet/resenas_con_clusters.parquet", index=False)#por si las moscas

In [26]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer


clusters_validos = sorted([c for c in df['cluster'].unique() if c != -1])


In [27]:
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=5,
)
 
tfidf_matrix = vectorizer.fit_transform(df['resena_limpia'])
vocab = vectorizer.get_feature_names_out()

In [28]:
def top_palabras_cluster(cluster_id, n=20):
    mask = (df['cluster'] == cluster_id).to_numpy()
    centroide = tfidf_matrix[mask].mean(axis=0)
    scores = np.asarray(centroide).flatten()
    top_idx = scores.argsort()[-n:][::-1]
    return [(vocab[i], round(scores[i], 4)) for i in top_idx]
 

In [39]:
resumen = []
 
for cid in clusters_validos:
    mask = df['cluster'] == cid
    n_resenas= mask.sum()
    pct_rec= df.loc[mask, 'recomendado'].mean() * 100
    palabras= top_palabras_cluster(cid)
 
    print(f" Cluster {cid}")
    print(f"Reseñas:{n_resenas:,}")
    print(f"% recomienda:{pct_rec:.1f}%")
    print(f"Top palabras:{[p for p, _ in palabras[:20]]}")
    print(f"Muestra:")
    muestra=df.loc[mask,'resena_limpia'].sample(min(50, n_resenas), random_state=31).tolist()
    for m in muestra:
        print(f"· {m[:300]}")
 
    resumen.append({
        'cluster':        cid,
        'n_resenas':      int(n_resenas),
        'pct_recomienda': round(pct_rec, 1),
        'top_palabras':   [p for p, _ in palabras[:20]],
        'emocion':        '',
    })

 Cluster 0
Reseñas:35,817
% recomienda:96.8%
Top palabras:['de', 'que', 'la', 'los', 'es', 'el', 'juego', 'en', 'un', 'juegos', 'de los', 'este', 'una', 'mejores', 'lo', 'no', 'los mejores', 'mejor', 'con', 'si']
Muestra:
· puedo decir con seguridad que life is strange es un buen juego, uno de sus puntos debiles es su poca jugabilidad, sin embargo lo compensa con muchos puntos fuertes, como sus entrañables personajes, o la historia envolvente, y un soundtrack muy bueno que incluso se convirtio en uno de mis favoritos d
· muy buen juego complementaron todo lo que faltaba en su versión original
· una obra maestra del survival horror. signalis toma conceptos y mecánicas de varios juegos clásicos del genero y añade también sus propias ideas innovadoras. el juego posee un estilo artístico de ciencia ficción a la antigua como el de películas de los años 70 (alien, odisea del espacio, etc) y una 
· despues del 5, este es el segundo mejor juego de sniper elite saga.
· perfecto para trios y fan

In [38]:
df

,resena_id,juego_id,titulo,resena_texto,recomendado,votos_utiles,votos_graciosos,puntuacion_ponderada,minutos_al_resenar,minutos_totales,fecha_creacion_unix,autor_num_resenas,recibido_gratis,escrito_acceso_anticipado,resena_limpia,cluster,emocion
0,222596476,445,Counter-Strike 2,"Juego LLENO de chiteros, con una comunidad de ...",0,1,0,0.523810,1231,1231.0,1775407333,1,0,0,"juego lleno de chiteros, con una comunidad de ...",2,frustracion_tecnica
1,222544394,445,Counter-Strike 2,Muy buen juego free to play pero si queres jug...,1,0,0,0.500000,3071,3352.0,1775354055,4,0,0,muy buen juego free to play pero si queres jug...,3,social_cooperativo
2,222459219,445,Counter-Strike 2,muchos peruanos y argentinos q lloran jugando ...,0,0,0,0.500000,301,380.0,1775271540,1,0,0,muchos peruanos y argentinos q lloran jugando ...,5,humor_intensidad_latina
3,222453511,445,Counter-Strike 2,Es complicado jugar cuando te toca un equipo d...,1,0,0,0.500000,58,119.0,1775265214,1,0,0,es complicado jugar cuando te toca un equipo d...,5,humor_intensidad_latina
4,222452532,445,Counter-Strike 2,"Es buen juego, pero esta lleno de chiteros! el...",1,0,0,0.500000,722,900.0,1775264163,1,0,0,"es buen juego, pero esta lleno de chiteros! el...",2,frustracion_tecnica
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295969,105747893,9604,Effie,Un juego plataformero 3D bastante simpático. N...,1,2,0,0.534884,291,291.0,1639371559,110,0,0,un juego plataformero 3d bastante simpático. n...,1,precio_valor
295970,93323934,9604,Effie,Compre este juego ya que sus imágenes que me r...,1,4,0,0.541667,267,267.0,1623019245,404,0,0,compre este juego ya que sus imágenes que me r...,1,precio_valor
295971,62882177,9834,Memoranda,Memoranda es un buen juego pero con algunos pu...,1,0,0,0.500000,313,313.0,1580678738,397,0,0,memoranda es un buen juego pero con algunos pu...,7,analisis_mixto
295972,212664552,7805,Noctropolis,¡Juro que solo quiero jugar un buen juego FMV!...,0,1,0,0.474372,215,215.0,1764995433,59,0,0,juro que solo quiero jugar un buen juego fmv! ...,8,analisis_narrativo_profundo


In [40]:
ETIQUETAS = {
    0: 'Gran calidad general con fallos puntuales y variabilidad',
    1: 'Bueno pero irregular: divertido, pero caro o incompleto',
    2: 'Problemas técnicos graves arruinan experiencia jugable',
    3: 'Entretenido casual, brilla más en multijugador',
    4: 'Obra maestra nostálgica, adictiva y emocional',
    5: 'Caótico, emocional, memeable y altamente polarizado',
    6: 'Buena calidad general en historia, arte y jugabilidad',
    7: 'Buen diseño, pero repetitivo y con desgaste',
    8: 'Análisis profundo con enfoque narrativo y artístico',
   -1: 'sin_cluster',
}

df['emocion'] = df['cluster'].map(ETIQUETAS)
df['emocion'].value_counts()

emocion
Buen diseño, pero repetitivo y con desgaste                 50243
Caótico, emocional, memeable y altamente polarizado         40367
Gran calidad general con fallos puntuales y variabilidad    35817
Obra maestra nostálgica, adictiva y emocional               32977
Buena calidad general en historia, arte y jugabilidad       32506
Problemas técnicos graves arruinan experiencia jugable      30435
Análisis profundo con enfoque narrativo y artístico         29680
Entretenido casual, brilla más en multijugador              21995
Bueno pero irregular: divertido, pero caro o incompleto     21954
Name: count, dtype: int64

In [47]:
#df = pd.DataFrame(resumen)
# df.to_csv("../datos/clusters_resumen_2000_v2.csv", index=False)
df.to_csv("../datos/Resenas_separdas.csv", index=False)
df.to_parquet("../datos/Resenas_separdas.parquet ", index=False)


In [43]:
df.columns

Index(['resena_id', 'juego_id', 'titulo', 'resena_texto', 'recomendado',
       'votos_utiles', 'votos_graciosos', 'puntuacion_ponderada',
       'minutos_al_resenar', 'minutos_totales', 'fecha_creacion_unix',
       'autor_num_resenas', 'recibido_gratis', 'escrito_acceso_anticipado',
       'resena_limpia', 'cluster', 'emocion'],
      dtype='str')

In [31]:
n_ruido = (df['cluster'] == -1).sum()
print(f"Sin cluster: {n_ruido:,} ({n_ruido/len(df)*100:.1f}%)")

Sin cluster: 232,313 (78.5%)


In [54]:
df[df['titulo'].str.contains('Marvel Rivals', case=False, na=False)]

,resena_id,juego_id,titulo,resena_texto,recomendado,votos_utiles,votos_graciosos,puntuacion_ponderada,minutos_al_resenar,minutos_totales,fecha_creacion_unix,autor_num_resenas,recibido_gratis,escrito_acceso_anticipado,resena_limpia,cluster,emocion


In [44]:

reducer_2d = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=31,
)

coords_2d = reducer_2d.fit_transform(embeddings)
#np.save("embeddings_umap_2d.npy", coords_2d)



c:\Users\ADRIA\OneDrive\Escritorio\Descargar_datos\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [45]:
df_plot = pd.DataFrame({
    'x':        coords_2d[:, 0],
    'y':        coords_2d[:, 1],
    'cluster':  df['cluster'].astype(str),
    'titulo':   df['titulo'],
    'reseña':   df['resena_limpia'].str[:120],
    'rec':      df['recomendado'].map({1: 'Sí', 0: 'No'}),
})

fig = px.scatter(
    df_plot,
    x='x', y='y',
    color='cluster',
    hover_data={'x': False, 'y': False, 'cluster': True, 'titulo': True, 'reseña': True, 'rec': True},
    width=950, height=650,
    opacity=0.4,
)

fig.write_html("vista_en_umap.html", auto_open=True)

In [34]:
for ms in [4, 5, 6, 7, 8]:
    c = hdbscan.HDBSCAN(
        min_cluster_size=2000,
        min_samples=ms,
        metric='euclidean',
        cluster_selection_method='eom',
    ).fit_predict(embeddings_umap)
    
    n_c = len(set(c)) - (1 if -1 in c else 0)
    n_r = (c == -1).sum()
    print(f"min_samples={ms} → clusters={n_c}, ruido={n_r:,} ({n_r/len(c)*100:.1f}%)")

min_samples=4 → clusters=9, ruido=228,423 (77.2%)
min_samples=5 → clusters=9, ruido=226,779 (76.6%)
min_samples=6 → clusters=9, ruido=226,554 (76.5%)
min_samples=7 → clusters=7, ruido=213,784 (72.2%)
min_samples=8 → clusters=7, ruido=209,451 (70.8%)


In [ ]:
# # acercar al mas cercano
# #mejro contnuar con mas revision de las reseñas para lo grar mejores resultados
# clusters_validos = sorted([c for c in df['cluster'].unique() if c != -1])
# centroides = np.array([embeddings_umap[df['cluster'] == c].mean(axis=0)for c in clusters_validos])

# mask_ruido = (df['cluster'] == -1).to_numpy()
# embeddings_ruido = embeddings_umap[mask_ruido]

# distancias = np.linalg.norm(embeddings_ruido[:, None, :] - centroides[None, :, :],axis=2)

# df.loc[mask_ruido, 'cluster'] = np.array(clusters_validos)[distancias.argmin(axis=1)]
# print(df['cluster'].value_counts().sort_index())